## Censo universitarios 2010 

In [26]:
# Paquetes
from pathlib import Path

import unicodedata
import pandas as pd
import pyreadstat
import matplotlib.pyplot as plt
import seaborn as sns

In [27]:
# Rutas de Censo 1996
ruta_2010 = Path(r"E:\BD\Censo Universitario\2010")

# Rutas de programas
ruta_correspondencias = Path(
    r"D:\Daniel_Pajita\1_Proyectos\Research\Educacion"
    r"\educacion-universitaria-peru\2_Datos"
    r"\Correspondencias_entre_clasificaciones"
)

#### Base de pregrado 2010

In [28]:
# Cargar todas las variables de pregrado: Censo 2010
pregrado_2010, meta_pregrado_2010 = pyreadstat.read_sav(str(ruta_2010 / "01_CENAUN_CED01_PRE-GRADO.sav"))
display(pregrado_2010.head())

,cap1_IDX_PERSONA,cap1_IDX_MARCO,cap1_FECREG,CCPAIS,cap1_PAIS_NOMBRE,CCDD,cap1_DPTO_NOMBRE,CCPP,cap1_PROV_NOMBRE,CCDI,...,MIGRA_EXT_3,MIGRA_EXT_4,MIGRA_EXT_5,MIGRA_EXT_6,MIGRA_EXT_6_ESP,LOTE,ASIG,SICODA_CCDI,FLAG_COMPLETO,tipo_univ
0,64564211033,68462571033,2010-03-30,4028,PERU,20,PIURA,06,SULLANA,01,...,0,0,1,0,,331006,,,1,2
1,64564513033,68462874033,2010-03-31,4028,PERU,20,PIURA,01,PIURA,01,...,0,0,0,0,,331002,,,1,2
2,64564554033,68462915033,2010-03-31,4028,PERU,20,PIURA,01,PIURA,04,...,,,,,,331003,,,1,2
3,64564588033,68462949033,2010-03-31,4028,PERU,20,PIURA,01,PIURA,07,...,0,0,1,0,,331029,,,1,2
4,64564890033,68463253033,2010-04-03,4028,PERU,20,PIURA,06,SULLANA,01,...,1,0,1,0,,331016,,,1,2


In [29]:
# Verificar si hay repeticiones de cap1_IDX_PERSONA
print("Repeticiones adicionales de cap1_IDX_PERSONA:",
    pregrado_2010["cap1_IDX_PERSONA"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .dropna()
    .duplicated()
    .sum()
)

Repeticiones adicionales de cap1_IDX_PERSONA: 124


In [30]:
# Seleccionar las variables de análisis: pregrado 2010
variables_pregrado_2010 = {
    "cap1_IDX_PERSONA": "id_persona_original",
    "cap1_CODUNI": "codigo_universidad",
    "cap1_UNIV_NOMBRE": "nombre_universidad",
    "tipo_univ": "tipo_universidad",
    "CODFAC": "codigo_facultad",
    "CODCARRPROF": "codigo_programa",
    "CODIGO": "codigo_carrera_universitaria",
    "ESPEC_NOMBRE": "nombre_programa",
    "CODCARRPROF_ESP": "nombre_programa_especificado",
    "SEXO": "sexo",
    "FDIA": "dia_nacimiento",
    "FMES": "mes_nacimiento",
    "FANIO": "anio_nacimiento_original",
    "RECCCPAIS": "codigo_pais_estudios",
    "RECPAIS_NOMBRE": "pais_estudios",
    "RECCCDD": "codigo_departamento_estudios",
    "RECDPTO_NOMBRE": "departamento_estudios",
    "RECCCPP": "codigo_provincia_estudios",
    "RECPROV_NOMBRE": "provincia_estudios",
    "RECCCDI": "codigo_distrito_estudios",
    "RECDIST_NOMBRE": "distrito_estudios",
    "NACCCPAIS": "codigo_pais_nacimiento",
    "cap2_PAIS_NOMBRE": "pais_nacimiento",
    "NACCCDD": "codigo_departamento_nacimiento",
    "cap2_DPTO_NOMBRE": "departamento_nacimiento",
    "NACCCPP": "codigo_provincia_nacimiento",
    "cap2_PROV_NOMBRE": "provincia_nacimiento",
    "NACCCDI": "codigo_distrito_nacimiento",
    "cap2_DIST_NOMBRE": "distrito_nacimiento",
}

pregrado_2010_analisis = (pregrado_2010[list(variables_pregrado_2010)].rename(columns=variables_pregrado_2010).copy())
pregrado_2010_analisis.insert(0, "id_fila", range(1, len(pregrado_2010_analisis) + 1))
pregrado_2010_analisis["nivel_academico"] = "Pregrado"

# Limpiar espacios en los campos de texto
for columna in pregrado_2010_analisis.select_dtypes(include=["object", "string"]).columns:
    pregrado_2010_analisis[columna] = (pregrado_2010_analisis[columna].astype("string").str.strip().replace("", pd.NA))

# Construir la fecha de nacimiento
pregrado_2010_analisis["fecha_nacimiento"] = pd.to_datetime(
    {
        "year": pd.to_numeric(pregrado_2010_analisis["anio_nacimiento_original"],errors="coerce",),
        "month": pd.to_numeric(pregrado_2010_analisis["mes_nacimiento"],errors="coerce",),
        "day": pd.to_numeric(pregrado_2010_analisis["dia_nacimiento"],errors="coerce",),
    },
    errors="coerce",
)

# Calcular la edad en años cumplidos al 31 de julio de 2010
fecha_referencia = pd.Timestamp("2010-07-31")
nacimiento = pregrado_2010_analisis["fecha_nacimiento"]

cumple_despues = (
    (nacimiento.dt.month > fecha_referencia.month) 
    | ((nacimiento.dt.month == fecha_referencia.month) & (nacimiento.dt.day > fecha_referencia.day))
)

pregrado_2010_analisis["edad"] = (fecha_referencia.year - nacimiento.dt.year - cumple_despues.astype(int)
                                  ).where(nacimiento <= fecha_referencia).astype("Int64")

# Construir los códigos territoriales de seis dígitos
# Su interpretación como ubigeo peruano depende del país registrado.
for lugar in ["estudios", "nacimiento"]:
    partes = []

    for division in ["departamento", "provincia", "distrito"]:
        columna = f"codigo_{division}_{lugar}"
        parte = (pd.to_numeric(pregrado_2010_analisis[columna], errors="coerce").astype("Int64").astype("string").str.zfill(2))
        partes.append(parte)

    pregrado_2010_analisis[f"codigo_territorial_{lugar}"] = (partes[0] + partes[1] + partes[2])

In [31]:
pregrado_2010_analisis.head()

,id_fila,id_persona_original,codigo_universidad,nombre_universidad,tipo_universidad,codigo_facultad,codigo_programa,codigo_carrera_universitaria,nombre_programa,nombre_programa_especificado,...,departamento_nacimiento,codigo_provincia_nacimiento,provincia_nacimiento,codigo_distrito_nacimiento,distrito_nacimiento,nivel_academico,fecha_nacimiento,edad,codigo_territorial_estudios,codigo_territorial_nacimiento
0,1,64564211033,033,UNIVERSIDAD PRIVADA CÉSAR VALLEJO,2,999,999,1707.0,<NA>,PSICOLOGIA,...,PIURA,06,SULLANA,01,SULLANA,Pregrado,1991-01-06,19,200601,200601
1,2,64564513033,033,UNIVERSIDAD PRIVADA CÉSAR VALLEJO,2,999,999,1707.0,<NA>,PSICOLOGIA,...,PIURA,04,MORROPON,01,CHULUCANAS,Pregrado,1980-11-21,29,200101,200401
2,3,64564554033,033,UNIVERSIDAD PRIVADA CÉSAR VALLEJO,2,005,999,1707.0,<NA>,PSICOLOGIA,...,LIMA,02,BARRANCA,02,PARAMONGA,Pregrado,1989-12-07,20,200104,150202
3,4,64564588033,033,UNIVERSIDAD PRIVADA CÉSAR VALLEJO,2,999,999,1707.0,<NA>,PSICOLOGIA,...,PIURA,01,PIURA,04,CASTILLA,Pregrado,1992-07-11,18,200101,200104
4,5,64564890033,033,UNIVERSIDAD PRIVADA CÉSAR VALLEJO,2,005,999,1707.0,<NA>,PSICOLOGIA,...,PIURA,06,SULLANA,01,SULLANA,Pregrado,1992-08-17,17,200101,200601


In [32]:
with pd.option_context("display.max_rows", None):
    display(pd.DataFrame({"Tipo": pregrado_2010_analisis.dtypes.astype(str),"No vacíos": pregrado_2010_analisis.count(),}).rename_axis("Variable"))

,Tipo,No vacíos
Variable,,
id_fila,int64,782970
id_persona_original,string,782970
codigo_universidad,string,782970
nombre_universidad,string,782970
tipo_universidad,string,782970
codigo_facultad,string,782970
codigo_programa,string,782970
codigo_carrera_universitaria,float64,782970
nombre_programa,string,734664


In [33]:
# Edad promedio por sexo y tipo de universidad: pregrado 2010
tabla_edad_pregrado = pd.pivot_table(
    pregrado_2010_analisis.assign(
        Sexo=pd.to_numeric(pregrado_2010_analisis["sexo"], errors="coerce").map({1: "Masculino", 2: "Femenino"}),
        Tipo_universidad=pd.to_numeric(pregrado_2010_analisis["tipo_universidad"], errors="coerce").map({1: "Pública", 2: "Privada"}),
    ),
    values="edad",index="Sexo",columns="Tipo_universidad",aggfunc="mean",margins=True,margins_name="TOTAL",
)

display(tabla_edad_pregrado.style.format("{:.2f}", na_rep="—"))

Tipo_universidad,Privada,Pública,TOTAL
Sexo,,,
Femenino,22.50,22.18,22.39
Masculino,23.76,23.17,23.50
TOTAL,23.10,22.74,22.96


#### Base de posgrado 2010

In [34]:
# Cargar todas las variables de posgrado: Censo 2010
posgrado_2010, meta_posgrado_2010 = pyreadstat.read_sav(str(ruta_2010 / "02_CENAUN_CED02_POST-GRADO.sav"))
display(posgrado_2010.head())

,cap1_IDX_PERSONA,cap1_IDX_MARCO,cap1_FECREG,CCPAIS,cap1_PAIS_NOMBRE,CCDD,cap1_DPTO_NOMBRE,CCPP,cap1_PROV_NOMBRE,CCDI,...,MIGRA_EXT_6,MIGRA_EXT_6_ESP,LOTE,ASIG,SICODA_CCDI,FLAG_COMPLETO,tipo_univ,COD_MAESTRIA,COD_ESPECIALIZACION,COD_DOCTORADO
0,64847186002,68747015002,2010-05-20,4028,PERU,02,ANCASH,01,HUARAZ,05,...,,,0022001,,,1,1,22,,
1,64847196002,68747025002,2010-05-16,4028,PERU,02,ANCASH,01,HUARAZ,01,...,,,0022001,,,1,1,06,,
2,64847207002,68747036002,2010-05-19,4028,PERU,02,ANCASH,01,HUARAZ,05,...,0,,0022001,,,1,1,22,,
3,66274004056,70201264056,2010-05-05,4014,ECUADOR,,,,,,...,,,0562001,,,1,2,27,,
4,64847215002,68747044002,2010-05-18,4028,PERU,02,ANCASH,20,YUNGAY,01,...,,,0022001,,,1,1,06,,


In [35]:
# Verificar si hay repeticiones de cap1_IDX_PERSONA
print("Repeticiones adicionales de cap1_IDX_PERSONA:",
    posgrado_2010["cap1_IDX_PERSONA"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .dropna()
    .duplicated()
    .sum()
)

Repeticiones adicionales de cap1_IDX_PERSONA: 312


In [36]:
# Seleccionar las variables de análisis: posgrado 2010
variables_posgrado_2010 = {
    "cap1_IDX_PERSONA": "id_persona_original",
    "cap1_CODUNI": "codigo_universidad",
    "cap1_UNIV_NOMBRE": "nombre_universidad",
    "tipo_univ": "tipo_universidad",
    "CODCARRPROF_2ESP": "codigo_programa_2esp",
    "ESPEC_NOMBRE_2ESP": "nombre_programa_2esp",
    "CODCARRPROF_2ESP_ESP": "nombre_especificado_2esp",
    "CODCARRPROF_MAES": "codigo_programa_maes",
    "ESPEC_NOMBRE_MAES": "nombre_programa_maes",
    "CODCARRPROF_MAES_ESP": "nombre_especificado_maes",
    "CODCARRPROF_DOCT": "codigo_programa_doct",
    "ESPEC_NOMBRE_DOCT": "nombre_programa_doct",
    "CODCARRPROF_DOCT_ESP": "nombre_especificado_doct",
    "SEXO": "sexo",
    "FDIA": "dia_nacimiento",
    "FMES": "mes_nacimiento",
    "FANIO": "anio_nacimiento_original",
    "RECCCPAIS": "codigo_pais_estudios",
    "cap1_RECPAIS_NOMBRE": "pais_estudios",
    "RECCCDD": "codigo_departamento_estudios",
    "cap1_RECDPTO_NOMBRE": "departamento_estudios",
    "RECCCPP": "codigo_provincia_estudios",
    "cap1_RECPROV_NOMBRE": "provincia_estudios",
    "RECCCDI": "codigo_distrito_estudios",
    "cap1_RECDIST_NOMBRE": "distrito_estudios",
    "NACCCPAIS": "codigo_pais_nacimiento",
    "cap2_PAIS_NOMBRE": "pais_nacimiento",
    "NACCCDD": "codigo_departamento_nacimiento",
    "cap2_DPTO_NOMBRE": "departamento_nacimiento",
    "NACCCPP": "codigo_provincia_nacimiento",
    "cap2_PROV_NOMBRE": "provincia_nacimiento",
    "NACCCDI": "codigo_distrito_nacimiento",
    "cap2_DIST_NOMBRE": "distrito_nacimiento",
}

posgrado_2010_analisis = (posgrado_2010[list(variables_posgrado_2010)].rename(columns=variables_posgrado_2010).copy())
posgrado_2010_analisis.insert(0, "id_fila", range(1, len(posgrado_2010_analisis) + 1))

# Limpiar espacios en los campos de texto
for columna in posgrado_2010_analisis.select_dtypes(include=["object", "string"]).columns:
    posgrado_2010_analisis[columna] = (posgrado_2010_analisis[columna].astype("string").str.strip().replace("", pd.NA))

# Identificar el nivel y seleccionar su programa
posgrado_2010_analisis["nivel_academico"] = "No especifica"

for columna in [
    "codigo_programa", "nombre_programa", "nombre_programa_especificado"
]:
    posgrado_2010_analisis[columna] = pd.Series(pd.NA, index=posgrado_2010_analisis.index, dtype="string")

for sufijo, nivel in [
    ("2esp", "Segunda especialidad"),
    ("maes", "Maestría"),
    ("doct", "Doctorado"),
]:
    tiene_programa = (posgrado_2010_analisis[f"codigo_programa_{sufijo}"].notna())

    posgrado_2010_analisis.loc[tiene_programa, "nivel_academico"] = nivel

    for destino, origen in [
        ("codigo_programa", f"codigo_programa_{sufijo}"),
        ("nombre_programa", f"nombre_programa_{sufijo}"),
        ("nombre_programa_especificado", f"nombre_especificado_{sufijo}"),
    ]:
        posgrado_2010_analisis.loc[tiene_programa, destino] = (posgrado_2010_analisis.loc[tiene_programa, origen].astype("string"))

# Construir la fecha de nacimiento
posgrado_2010_analisis["fecha_nacimiento"] = pd.to_datetime(
    {
        "year": pd.to_numeric(posgrado_2010_analisis["anio_nacimiento_original"],errors="coerce",),
        "month": pd.to_numeric(posgrado_2010_analisis["mes_nacimiento"], errors="coerce"),
        "day": pd.to_numeric(posgrado_2010_analisis["dia_nacimiento"], errors="coerce"),
    },
    errors="coerce",
)

# Calcular la edad en años cumplidos al 31 de julio de 2010
fecha_referencia = pd.Timestamp("2010-07-31")
nacimiento = posgrado_2010_analisis["fecha_nacimiento"]

cumple_despues = (
    (nacimiento.dt.month > fecha_referencia.month)
    | ((nacimiento.dt.month == fecha_referencia.month) & (nacimiento.dt.day > fecha_referencia.day)
    )
)

posgrado_2010_analisis["edad"] = (fecha_referencia.year - nacimiento.dt.year - cumple_despues.astype(int)
).where(nacimiento <= fecha_referencia).astype("Int64")

# Construir los códigos territoriales de seis dígitos
# Su interpretación como ubigeo peruano depende del país registrado.
for lugar in ["estudios", "nacimiento"]:
    partes = []

    for division in ["departamento", "provincia", "distrito"]:
        columna = f"codigo_{division}_{lugar}"
        parte = (pd.to_numeric(posgrado_2010_analisis[columna], errors="coerce").astype("Int64").astype("string").str.zfill(2))
        partes.append(parte)

    posgrado_2010_analisis[f"codigo_territorial_{lugar}"] = (partes[0] + partes[1] + partes[2])

In [37]:
posgrado_2010_analisis.head()

,id_fila,id_persona_original,codigo_universidad,nombre_universidad,tipo_universidad,codigo_programa_2esp,nombre_programa_2esp,nombre_especificado_2esp,codigo_programa_maes,nombre_programa_maes,...,codigo_distrito_nacimiento,distrito_nacimiento,nivel_academico,codigo_programa,nombre_programa,nombre_programa_especificado,fecha_nacimiento,edad,codigo_territorial_estudios,codigo_territorial_nacimiento
0,1,64847186002,002,UNIVERSIDAD NACIONAL SANTIAGO ANTÚNEZ DE MAYOLO,1,<NA>,<NA>,<NA>,003,CIENCIAS PENALES,...,13,JESUS MARIA,Maestría,003,CIENCIAS PENALES,<NA>,1978-09-01,31,020105,150113
1,2,64847196002,002,UNIVERSIDAD NACIONAL SANTIAGO ANTÚNEZ DE MAYOLO,1,<NA>,<NA>,<NA>,005,DIRECCIÓN DE LA CONSTRUCCIÓN,...,01,HUARAZ,Maestría,005,DIRECCIÓN DE LA CONSTRUCCIÓN,<NA>,1974-10-26,35,020101,020101
2,3,64847207002,002,UNIVERSIDAD NACIONAL SANTIAGO ANTÚNEZ DE MAYOLO,1,<NA>,<NA>,<NA>,004,DERECHO CIVIL Y COMERCIAL,...,04,CHINGAS,Maestría,004,DERECHO CIVIL Y COMERCIAL,<NA>,1977-10-21,32,020105,020304
3,4,66274004056,056,UNIVERSIDAD PERUANA UNIÓN,2,<NA>,<NA>,<NA>,007,TEOLOGÍA,...,<NA>,<NA>,Maestría,007,TEOLOGÍA,<NA>,1979-09-17,30,150118,<NA>
4,5,64847215002,002,UNIVERSIDAD NACIONAL SANTIAGO ANTÚNEZ DE MAYOLO,1,<NA>,<NA>,<NA>,005,DIRECCIÓN DE LA CONSTRUCCIÓN,...,01,YUNGAY,Maestría,005,DIRECCIÓN DE LA CONSTRUCCIÓN,<NA>,1975-12-18,34,020101,022001


In [38]:
# Edad promedio por sexo, tipo de universidad y nivel académico: posgrado 2010
tabla_edad_posgrado = pd.pivot_table(
    posgrado_2010_analisis.assign(
        Sexo=pd.to_numeric(posgrado_2010_analisis["sexo"], errors="coerce").map({1: "Masculino", 2: "Femenino"}),
        Tipo_universidad=pd.to_numeric(posgrado_2010_analisis["tipo_universidad"], errors="coerce").map({1: "Pública", 2: "Privada"}),
    ),
    values="edad",index=["nivel_academico", "Sexo"],columns="Tipo_universidad",aggfunc="mean",margins=True,margins_name="TOTAL",
)

display(tabla_edad_posgrado.style.format("{:.2f}", na_rep="—"))

In [39]:
# Tabla resumen de pregrado y posgrado: Censo 2010
datos_tabla = pd.concat(
    [
        pregrado_2010_analisis[["nivel_academico", "sexo", "tipo_universidad", "edad"]],
        posgrado_2010_analisis[["nivel_academico", "sexo", "tipo_universidad", "edad"]],
    ],
    ignore_index=True,
)

datos_tabla["Sexo"] = pd.to_numeric(datos_tabla["sexo"], errors="coerce").map({1: "Masculino", 2: "Femenino"})
datos_tabla["Tipo_universidad"] = pd.to_numeric(datos_tabla["tipo_universidad"], errors="coerce").map({1: "Pública", 2: "Privada"})

niveles = ["Pregrado", "Segunda especialidad", "Maestría", "Doctorado"]
datos_tabla = datos_tabla.loc[datos_tabla["nivel_academico"].isin(niveles)]

# Resumir cada combinación, incluyendo los totales
resultados = {}

for sexo in ["Masculino", "Femenino", "TOTAL"]:
    for tipo in ["Pública", "Privada", "TOTAL"]:
        grupo = datos_tabla

        if sexo != "TOTAL":
            grupo = grupo.loc[grupo["Sexo"] == sexo]
        if tipo != "TOTAL":
            grupo = grupo.loc[grupo["Tipo_universidad"] == tipo]

        resumen = (grupo.groupby("nivel_academico").agg(Edad_promedio=("edad", "mean"),Estudiantes=("edad", "size"),).reindex(niveles))
        resumen.loc["TOTAL"] = [grupo["edad"].mean(),len(grupo),]
        resultados[(sexo, tipo)] = resumen

tabla_edad_nivel = pd.concat(resultados, axis=1)
tabla_edad_nivel.columns.names = ["Sexo", "Tipo de universidad", "Medida"]
tabla_edad_nivel.index.name = "Nivel académico"

display(tabla_edad_nivel.style.format(
    {
        columna: ("{:.1f}" if columna[2] == "Edad_promedio" else "{:,.0f}")
        for columna in tabla_edad_nivel.columns
    },
    na_rep="—",
))

### Programas del censo 2010

In [40]:
# Programas únicos de pregrado
programas_pregrado_2010 = (
    pregrado_2010_analisis[["nombre_programa", "nivel_academico"]]
    .drop_duplicates()
    .sort_values(["nivel_academico", "nombre_programa"])
    .reset_index(drop=True)
)

# Programas únicos de posgrado
programas_posgrado_2010 = (
    posgrado_2010_analisis[["nombre_programa", "nivel_academico"]]
    .drop_duplicates()
    .sort_values(["nivel_academico", "nombre_programa"])
    .reset_index(drop=True)
)

In [41]:
programas_pregrado_2010.head()

,nombre_programa,nivel_academico
0,ACUICULTURA,Pregrado
1,ADM. TURISTICA Y HOTELERA,Pregrado
2,ADMINISTRACION,Pregrado
3,ADMINISTRACION DE EMPRESAS,Pregrado
4,ADMINISTRACION DE EMPRESAS TURISTICAS,Pregrado


In [42]:
programas_posgrado_2010.head()

,nombre_programa,nivel_academico
0,(PH.D.) INTERNACIONAL EN LIDERAZGO,Doctorado
1,. ENFERMERIA,Doctorado
2,ADMINISTRACION,Doctorado
3,ADMINISTRACION ESTRATEGICA DE EMPRESAS,Doctorado
4,ADMINISTRACIÓN,Doctorado


In [43]:
# Combinar los listados de programas
programas_2010 = pd.concat([programas_pregrado_2010, programas_posgrado_2010],ignore_index=True,)

# Estandarizar los nombres
def estandarizar_programa(texto):
    if pd.isna(texto):
        return pd.NA

    texto = unicodedata.normalize("NFD", str(texto).upper())
    texto = "".join(
        caracter for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )
    texto = "".join(
        caracter if caracter.isalnum() else " "
        for caracter in texto
    )
    return " ".join(texto.split())


programas_2010["nombre_programa"] = (programas_2010["nombre_programa"].map(estandarizar_programa))

# Conservar combinaciones únicas de programa y nivel
programas_2010 = (
    programas_2010
    .drop_duplicates(subset=["nombre_programa", "nivel_academico"])
    .sort_values(["nivel_academico", "nombre_programa"])
    .reset_index(drop=True)
)

# Exportar el insumo para clasificación
programas_2010.to_csv(ruta_correspondencias / "programas_censo2010.txt",sep="|",index=False,encoding="utf-8-sig",)

In [44]:
programas_2010

,nombre_programa,nivel_academico
0,ADMINISTRACION,Doctorado
1,ADMINISTRACION DE LA EDUCACION,Doctorado
2,ADMINISTRACION ESTRATEGICA DE EMPRESAS,Doctorado
3,ADMINISTRACION Y DIRECCION DE EMPRESAS,Doctorado
4,ADMINSTRACION,Doctorado
...,...,...
1360,SEP EN INVESTIGACION INNOVACION Y ACREDITACION...,Segunda especialidad
1361,SEP EN TECNOLOGIA COMPUTACION E INFORMATICA ED...,Segunda especialidad
1362,SISTEMAS DE POTENCIA,Segunda especialidad
1363,TELEMATICA,Segunda especialidad
